In [24]:
from langchain_classic.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [ ]:
model = ChatOpenAI(
    model='gpt-3.5-turbo', 
    temperature=0.7,
    max_completion_tokens=50
)

In [43]:
inputs = ["python", "java"]

chat_1 = ChatPromptTemplate.from_template(
    "suggest three {pl} books. answer only by listing"
)

chat_2 = ChatPromptTemplate.from_template(
    "suggest three {pl} projects. answer only by listing"
)

In [11]:
str_parser = StrOutputParser()

In [46]:
chain_books = chat_1 | model | str_parser

In [47]:
chain_project = chat_2 | model | str_parser

In [50]:
chain_parallel = RunnableParallel({'books':chain_books, 'projects':chain_project})
chain_parallel.invoke('python')

{'books': '1. "Automate the Boring Stuff with Python" by Al Sweigart\n2. "Python Crash Course" by Eric Matthes\n3. "Fluent Python" by Luciano Ramalho',
 'projects': '1. A web scraping tool to extract data from websites and store it in a database.\n2. A chatbot using natural language processing to answer user queries.\n3. A data visualization tool to create interactive graphs and charts from datasets.'}

In [23]:
chain_parallel.get_graph().print_ascii()

            +-------------------------------+              
            | Parallel<books,projects>Input |              
            +-------------------------------+              
                   ***               ***                   
                ***                     ***                
              **                           **              
+--------------------+              +--------------------+ 
| ChatPromptTemplate |              | ChatPromptTemplate | 
+--------------------+              +--------------------+ 
           *                                   *           
           *                                   *           
           *                                   *           
    +------------+                      +------------+     
    | ChatOpenAI |                      | ChatOpenAI |     
    +------------+                      +------------+     
           *                                   *           
           *                            

RUNNABLE PARALLEL + RUNNABLE

In [44]:
chain_parallel = RunnableParallel({'books':chain_books, 'projects':chain_project})
chain_parallel.batch(inputs)

chat_3 = ChatPromptTemplate.from_template(
    "how long would it take to complete {books} and {projects}"
)

chain = chain_parallel | {'books':RunnablePassthrough(), 'projects':RunnablePassthrough()} | chat_3 | model
chain.get_graph().print_ascii()

            +-------------------------------+              
            | Parallel<books,projects>Input |              
            +-------------------------------+              
                   ***               ***                   
                ***                     ***                
              **                           **              
+--------------------+              +--------------------+ 
| ChatPromptTemplate |              | ChatPromptTemplate | 
+--------------------+              +--------------------+ 
           *                                   *           
           *                                   *           
           *                                   *           
    +------------+                      +------------+     
    | ChatOpenAI |                      | ChatOpenAI |     
    +------------+                      +------------+     
           *                                   *           
           *                            